## Extração de características

Este notebook carrega os dados de `data/interim` e `data/processed` e extrai um vetor de características por trial. O resultado é uma matriz no formato (trials, características) com os rótulos, o participante e a sessão de cada linha salvos em `data/features`.

As características são calculadas em cada canal de interesse e em cada trial. Elas foram escolhidas com base nas bandas alfa e beta, usadas por Nieto et al.

In [ ]:
# Imports
import numpy as np
from pathlib import Path
from scipy.signal import welch

# List with every subject (S1-S10)
N_S = list(range(1, 11))

# sub-09 ses-01 and sub-10 ses-03 are truncated in the OpenNeuro datasets
EXCLUDE = {9, 10}
SUBJECTS = [i for i in N_S if i not in EXCLUDE]
CLASSES = {0: "up", 1: "down", 2: "right", 3: "left"}

# EEG electrodes used by Nieto et al. in the PSD comparisons (Fig. 9)
EEG_ELECTRODES = {
    "A": [4, 5, 10, 13, 17, 19, 20, 21, 22, 26, 30, 32],
    "B": [7, 16, 22, 24, 29],
    "D": [10, 19, 21, 26],
}

# Time window of interest
t_start, t_end = 1.5, 3.5

# Sampling rate
fs = 256

# Directories with the np arrays generated in 02-load, and where the features are saved
interim_dir = Path.cwd().parent / "data" / "interim"
processed_dir = Path.cwd().parent / "data" / "processed"

# Directory for the features array
features_dir = Path.cwd().parent / "data" / "features"
features_dir.mkdir(parents=True, exist_ok=True)

### Carregamento dos dados

Os dados são carregados no formato (classes, trials, channels, time points). Apenas os eletrodos de interesse, os mesmos utilizados em `03-visualization`, são considerados, com a janela de tempo de 1.5 a 3.5s, como em Nieto et al.

In [ ]:
# Data in (classes, trials, channels, time points) format
X_eeg = np.load(processed_dir / "X_eeg.npy", mmap_mode="r")
subj = np.load(processed_dir / "subj.npy")
ses = np.load(processed_dir / "ses.npy")
names_eeg = list(np.load(interim_dir / "S1_eeg_channels.npy"))
times = np.load(interim_dir / "S1_times.npy")

ch_eeg = [f"{letter}{n}" for letter, nums in EEG_ELECTRODES.items() for n in nums]
idx_eeg = [names_eeg.index(c) for c in ch_eeg]
start, end = round(t_start * fs), round(t_end * fs)

# Keep only the channels of interest and the time window
X = np.asarray(X_eeg[:, :, idx_eeg][:, :, :, start:end], dtype=np.float64) * 1e6  # volts -> microvolts
print("fs:", fs, "\nepoch time range:", times[0], times[-1])
print("Data after selection: ", X.shape)

# Reshape classes and trials into one axis
n_cls, n_trials = X.shape[:2]
X = X.reshape(n_cls * n_trials, *X.shape[2:])

# One label, subject and session per sample
y = np.repeat(np.arange(n_cls), n_trials)
subj_flat = subj.reshape(-1)
ses_flat = ses.reshape(-1)

print(X.shape, y.shape, subj_flat.shape, ses_flat.shape)

### Características

Foram consideradas as seguintes características:

#### Domínio do Tempo
- Variance of EMG (VAR)

    > $ \frac{1}{N-1}\sum_{i=1}^{N}x_i^2 $

- Root Mean Square (RMS):
    
    > $ \sqrt{\frac{1}{N}\sum_{i=1}^{N}|x_i|^2} $

- Waveform Length (WL):

    > $ \sum_{i=1}^{N-1}|x_{i+1} - x_i| $

- Zero Crossing (ZC):

    > $ \sum_{i=1}^{N-1}sgn(x_i, x_{i+1}) $

    > $ sgn(x_i, x_{i+1}) = \begin{cases} 1 & \text{if } x_i \cdot x_{i+1} < 0 \\ 0 & \text{otherwise} \end{cases} $

In [ ]:
# Time domain features
def var(x):
    return np.var(x, axis=-1, ddof=1)

def rms(x):
    return np.sqrt(np.mean(x ** 2, axis=-1))

def wl(x):
    return np.sum(np.abs(np.diff(x, axis=-1)), axis=-1)

def zc(x):
    return np.sum(x[..., :-1] * x[..., 1:] < 0, axis=-1)

#### Domínio da frequência

As características no domínio da frequência são calculadas a partir da faixa de 1 a 45 Hz:

- Median Frequency (FMD)

    > $ \frac{1}{2}\sum_{j=1}^{M}PSD_j $

- Mean Frequency (FMN)

    > $ \sum_{j=1}^{M}f_j PSD_j \Big/ \sum_{j=1}^{M}PSD_j $
    
    > $ f_j = j * SampleRate \Big/ 2 * M $

- Modified Median Frequency (MMDF)

    > $ \frac{1}{2}\sum_{j=1}^{M}A_j $
    
    > $ A_j = Amplitude\ do\ espectro\ j $

- Modified Frequency Mean (MMNF)

    > $ \sum_{j=1}^{M}f_jAj \Big/ \sum_{j=1}^{M}Aj $

- Band Potency (BP):

    > $ BP_b = \log_{10}\left(\frac{1}{|b|}\sum_{f_j \in b}PSD_j\right) $

In [ ]:
# Frequency domain features
bands = {"theta": (4, 8), "alpha": (8, 12), "beta": (12, 30), "gamma": (30, 45)}

freqs, psd = welch(X, fs=fs, nperseg=256, axis=-1)

selected_freqs = (freqs >= 1) & (freqs <= 45)
freqs = freqs[selected_freqs]
psd = psd[..., selected_freqs]
amp = np.sqrt(psd)

def fmd(psd, f):
    csum = np.cumsum(psd, axis=-1)
    return f[np.argmax(csum >= csum[..., -1:] / 2, axis=-1)]

def fmn(psd, f):
    return np.sum(psd * f, axis=-1) / np.sum(psd, axis=-1)

def mmdf(amp, f):
    csum = np.cumsum(amp, axis=-1)
    return f[np.argmax(csum >= csum[..., -1:] / 2, axis=-1)]

def mmnf(amp, f):
    return np.sum(amp * f, axis=-1) / np.sum(amp, axis=-1)

def band_power(psd, freqs, fmin, fmax):
    band = (freqs >= fmin) & (freqs < fmax)
    return np.log10(psd[..., band].mean(axis=-1))

### Vetor de características

Para cada característica, é construída uma matriz no formato (samples, channels). As matrizes são unidas no formato (samples, channels × features), de modo que cada coluna seja uma característica de um canal.

In [ ]:
features = {
    "var": var(X),
    "rms": rms(X),
    "wl": wl(X),
    "zc": zc(X),
    "fmd": fmd(psd, freqs),
    "fmn": fmn(psd, freqs),
    "mmdf": mmdf(amp, freqs),
    "mmnf": mmnf(amp, freqs),
    **{f"bp_{name}": band_power(psd, freqs, *limits) for name, limits in bands.items()},
}

feature_names = list(features)

# (features, samples, channels) to (samples, channels, features)
feat = np.stack([features[f] for f in feature_names]).transpose(1, 2, 0)

# (samples, channels * features)
X_features = feat.reshape(feat.shape[0], -1)
column_names = [f"{ch}_{f}" for ch in ch_eeg for f in feature_names]

print("Features per channel:", feature_names)
print("Feature matrix:", X_features.shape)
print("Classification shape:", feat.shape)

In [ ]:
# Saves the feature matrix, labels and the subject and session of each sample
np.save(features_dir / "X_features.npy", X_features)
np.save(features_dir / "y.npy", y)
np.save(features_dir / "subj_flat.npy", subj_flat)
np.save(features_dir / "ses_flat.npy", ses_flat)
np.save(features_dir / "feature_names.npy", np.asarray(column_names))

print(f"Saved {X_features.shape[0]} samples with {X_features.shape[1]} features")